# TP53 Structure Visualisation Notebook
## Multi-Agent Bioinformatics Platform · Gemma 4 Hackathon

This notebook provides interactive 3D visualisation of TP53 structure analysis results.  
Runs 100% locally. No internet required after initial model download.

**Views:**
1. 3D protein structure with mutation sites highlighted (py3Dmol)
2. ESM-2 residue embedding space (Plotly 3D scatter)
3. pLDDT confidence scores per residue
4. Domain architecture with mutation overlay
5. Wildtype vs Mutant structural comparison
6. Gemma 4 RAG narration

In [1]:
# Setup
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import json
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import py3Dmol
from IPython.display import display, HTML

# Load structure data
ACCESSION = 'NM_000546'  # Change this to your accession
DATA_PATH = Path(f'../../data/structure_outputs/{ACCESSION}_structure_data.json')

if DATA_PATH.exists():
    with open(DATA_PATH) as f:
        data = json.load(f)
    print(f'✓ Loaded structure data for {ACCESSION}')
    print(f'  Mutations: {len(data["mutations"])}')
    print(f'  Embeddings: {len(data["embedding_coords"])} residues')
    print(f'  Model: {data["model_used"]}')
else:
    print(f'No data found at {DATA_PATH}')
    print('Run first: python main.py visualise --accession NM_000546')
    data = None


No data found at ..\..\data\structure_outputs\NM_000546_structure_data.json
Run first: python main.py visualise --accession NM_000546


# ── 1. 3D Protein Structure (py3Dmol) ─────────────────────
# Displays wildtype structure with mutation sites coloured

PDB_PATH = Path(f'../../data/structure_outputs/{ACCESSION}_wildtype.pdb')

if PDB_PATH.exists():
    pdb_str = PDB_PATH.read_text()
    mutations = data['mutations'] if data else []
    hotspot_positions = [m['position'] for m in mutations if m['is_hotspot']]
    other_mut_positions = [m['position'] for m in mutations if not m['is_hotspot']]
    
    view = py3Dmol.view(width=900, height=500)
    view.addModel(pdb_str, 'pdb')
    
    # Cartoon representation coloured by secondary structure
    view.setStyle({'cartoon': {'color': 'spectrum', 'opacity': 0.85}})
    
    # Highlight hotspot mutations in red with spheres
    for pos in hotspot_positions:
        view.addStyle(
            {'resi': pos},
            {'sphere': {'color': '#ff3366', 'radius': 1.2}}
        )
        view.addLabel(
            next(m['label'] for m in mutations if m['position'] == pos),
            {'position': {'resi': pos}, 'fontSize': 10,
             'fontColor': '#ff3366', 'backgroundColor': 'white',
             'backgroundOpacity': 0.7}
        )
    
    # Non-hotspot mutations in orange
    for pos in other_mut_positions:
        view.addStyle(
            {'resi': pos},
            {'sphere': {'color': '#ff6b35', 'radius': 0.9}}
        )
    
    # Highlight DNA-binding domain in teal
    view.addStyle(
        {'resi': '94-292'},
        {'cartoon': {'color': '#4ECDC4', 'opacity': 0.9}}
    )
    
    view.zoomTo()
    view.setBackgroundColor('#050a0e')
    view.spin(True)  # Gentle rotation
    
    display(HTML(f'<h3 style="font-family:monospace;color:#00d4ff">TP53 Wildtype Structure · {ACCESSION}</h3>'))
    display(HTML('<p style="font-family:monospace;font-size:12px;color:#666">🔴 Hotspot mutations &nbsp; 🟠 Non-hotspot mutations &nbsp; 🩵 DNA-binding domain (94-292)</p>'))
    view.show()
else:
    print(f'PDB file not found: {PDB_PATH}')
    print('Run: python main.py visualise first')

# ── 2. Wildtype vs Mutant Side-by-Side ────────────────────

if data and data.get('mutations'):
    first_mutant = data['mutations'][0]['label']
    mut_pdb_path = Path(f'../../data/structure_outputs/{ACCESSION}_{first_mutant}.pdb')
    
    if PDB_PATH.exists() and mut_pdb_path.exists():
        wt_pdb = PDB_PATH.read_text()
        mut_pdb = mut_pdb_path.read_text()
        
        # Side by side viewers
        display(HTML(f'''
        <div style="display:flex;gap:16px;font-family:monospace">
            <div style="flex:1;text-align:center">
                <h4 style="color:#00d4ff;margin-bottom:8px">Wildtype p53</h4>
            </div>
            <div style="flex:1;text-align:center">
                <h4 style="color:#ff3366;margin-bottom:8px">Mutant: {first_mutant}</h4>
            </div>
        </div>
        '''))
        
        for label, pdb_str, color in [(f'Wildtype', wt_pdb, '#4ECDC4'), (first_mutant, mut_pdb, '#ff3366')]:
            v = py3Dmol.view(width=440, height=400)
            v.addModel(pdb_str, 'pdb')
            v.setStyle({'cartoon': {'color': 'spectrum'}})
            # Highlight the mutation site
            mut_pos = data['mutations'][0]['position']
            v.addStyle({'resi': mut_pos}, {'sphere': {'color': color, 'radius': 1.5}})
            v.zoomTo({'resi': f'{max(1,mut_pos-20)}-{min(393,mut_pos+20)}'})
            v.setBackgroundColor('#0d1821')
            v.show()
    else:
        print('PDB files not found. Run analysis first.')
else:
    print('No mutations in data.')

# ── 3. ESM-2 Embedding Space (3D Scatter) ─────────────────
# Visualises where each residue sits in ESM-2 embedding space
# Mutations that shift position significantly = greater functional impact

if data and data.get('embedding_coords') and len(data['embedding_coords']) > 0:
    coords = np.array(data['embedding_coords'])
    mutations = data['mutations']
    domains = data['domain_annotations']
    n = len(coords)
    
    # Colour each residue by domain
    colors = []
    domain_labels = []
    for i in range(n):
        residue = i + 1
        domain = next((d for d in domains if d['start'] <= residue <= d['end']), None)
        colors.append(domain['color'] if domain else '#4a7a8a')
        domain_labels.append(domain['short'] if domain else 'other')
    
    # Override mutation positions
    mut_positions = {m['position']: m for m in mutations}
    sizes = [14 if (i+1) in mut_positions else 4 for i in range(n)]
    for i in range(n):
        if (i+1) in mut_positions:
            m = mut_positions[i+1]
            colors[i] = '#ff3366' if m['is_hotspot'] else '#ff6b35'
    
    fig = go.Figure()
    
    # Main residue cloud
    fig.add_trace(go.Scatter3d(
        x=coords[:,0], y=coords[:,1], z=coords[:,2],
        mode='markers',
        name='WT Residues',
        marker=dict(size=sizes, color=colors, opacity=0.8,
                   line=dict(width=0)),
        text=[f'Res {i+1} · {domain_labels[i]}' + 
              (f' · {mut_positions[i+1]["label"]} ⚠' if (i+1) in mut_positions else '')
              for i in range(n)],
        hovertemplate='%{text}<extra></extra>',
    ))
    
    # Add mutant traces
    mutant_palette = ['#00d4ff', '#a29bfe', '#00ff88']
    for idx, (label, mcoords_list) in enumerate(data.get('mutant_embedding_coords', {}).items()):
        if not mcoords_list:
            continue
        mcoords = np.array(mcoords_list)
        fig.add_trace(go.Scatter3d(
            x=mcoords[:,0], y=mcoords[:,1], z=mcoords[:,2],
            mode='markers',
            name=f'Mutant: {label}',
            marker=dict(size=3, color=mutant_palette[idx % 3],
                       opacity=0.45, symbol='diamond',
                       line=dict(width=0)),
        ))
    
    fig.update_layout(
        title=dict(text=f'ESM-2 Residue Embeddings · {ACCESSION}',
                  font=dict(family='monospace', color='#00d4ff', size=14)),
        paper_bgcolor='#050a0e',
        plot_bgcolor='#050a0e',
        font=dict(family='monospace', color='#4a7a8a', size=10),
        scene=dict(
            bgcolor='#050a0e',
            xaxis=dict(gridcolor='#1a2840', zerolinecolor='#1a2840'),
            yaxis=dict(gridcolor='#1a2840', zerolinecolor='#1a2840'),
            zaxis=dict(gridcolor='#1a2840', zerolinecolor='#1a2840'),
        ),
        width=900, height=600,
        legend=dict(font=dict(color='#4a7a8a'), bgcolor='rgba(0,0,0,0)'),
    )
    
    fig.show()
    print('\n🔴 Red = hotspot mutations  🟠 Orange = non-hotspot  Other colours = protein domains')
    print('Translucent points = mutant sequence embeddings (shift = mutation effect)')
else:
    print('Embedding data not available. Install fair-esm: pip install fair-esm')

# ── 4. pLDDT Confidence Scores ────────────────────────────
# Per-residue structure prediction confidence from ESMFold
# >90 = high confidence (dark blue)  <50 = low confidence (red)

if data and data.get('plddt_scores') and len(data['plddt_scores']) > 0:
    scores = np.array(data['plddt_scores'])
    residues = list(range(1, len(scores)+1))
    
    # Colour by confidence tier
    colors = ['#3498db' if s >= 90 else '#2ecc71' if s >= 70
              else '#f39c12' if s >= 50 else '#e74c3c'
              for s in scores]
    
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=residues, y=scores,
        marker_color=colors,
        name='pLDDT',
        hovertemplate='Residue %{x}: %{y:.1f}<extra></extra>',
    ))
    
    # Add mutation position markers
    if data.get('mutations'):
        for m in data['mutations']:
            fig.add_vline(
                x=m['position'], line_width=1.5,
                line_color='#ff3366' if m['is_hotspot'] else '#ff6b35',
                annotation_text=m['label'],
                annotation_font_color='#ff3366' if m['is_hotspot'] else '#ff6b35',
                annotation_font_size=9,
            )
    
    # Add domain shading
    for d in data.get('domain_annotations', []):
        fig.add_vrect(
            x0=d['start'], x1=d['end'],
            fillcolor=d['color'], opacity=0.07,
            annotation_text=d['short'],
            annotation_font_size=8,
            annotation_font_color=d['color'],
        )
    
    fig.update_layout(
        title='ESMFold pLDDT Confidence per Residue',
        xaxis_title='Residue Position',
        yaxis_title='pLDDT Score',
        yaxis=dict(range=[0, 100]),
        paper_bgcolor='#050a0e',
        plot_bgcolor='#0d1821',
        font=dict(family='monospace', color='#4a7a8a', size=10),
        xaxis=dict(gridcolor='#1a2840'),
        yaxis_gridcolor='#1a2840',
        width=950, height=300,
    )
    
    fig.show()
    print('🔵 >90 (very high) 🟢 70-90 (high) 🟡 50-70 (moderate) 🔴 <50 (low confidence — disordered regions)')
else:
    print('pLDDT scores not available (requires ESMFold prediction)')

# ── 5. Domain Architecture + Mutations ───────────────────

if data:
    domains = data.get('domain_annotations', [])
    mutations = data.get('mutations', [])
    
    fig = go.Figure()
    
    # Draw domain bars
    for i, d in enumerate(domains):
        fig.add_trace(go.Bar(
            x=[d['end'] - d['start']],
            y=[0],
            base=[d['start']],
            orientation='h',
            marker_color=d['color'],
            marker_opacity=0.8,
            name=d['name'],
            text=d['short'],
            textposition='inside',
            insidetextanchor='middle',
            hovertemplate=f"{d['name']}<br>{d['start']}–{d['end']}<extra></extra>",
            width=0.4,
        ))
    
    # Draw mutation markers
    for m in mutations:
        fig.add_trace(go.Scatter(
            x=[m['position']],
            y=[0.35],
            mode='markers+text',
            marker=dict(
                symbol='triangle-down',
                size=14,
                color='#ff3366' if m['is_hotspot'] else '#ff6b35',
                line=dict(width=0),
            ),
            text=[m['label']],
            textposition='top center',
            textfont=dict(size=9, color='#ff3366' if m['is_hotspot'] else '#ff6b35',
                         family='monospace'),
            showlegend=False,
            hovertemplate=f"{m['label']} · {m['domain']} · {m['functional_impact']}<extra></extra>",
        ))
    
    fig.update_layout(
        title='p53 Domain Architecture with Mutation Overlay',
        barmode='overlay',
        xaxis=dict(range=[0, 400], title='Residue Position', gridcolor='#1a2840'),
        yaxis=dict(range=[-0.5, 0.8], showticklabels=False, gridcolor='#1a2840'),
        paper_bgcolor='#050a0e',
        plot_bgcolor='#0d1821',
        font=dict(family='monospace', color='#4a7a8a', size=10),
        width=950, height=220,
        showlegend=True,
        legend=dict(orientation='h', y=-0.3, font=dict(size=9, color='#4a7a8a'),
                   bgcolor='rgba(0,0,0,0)'),
    )
    
    fig.show()

# ── 6. Gemma 4 Structural Narration ───────────────────────

if data and data.get('llm_narration'):
    display(HTML('''
    <div style="background:#0d1821;border:1px solid #1a2840;border-left:3px solid #00d4ff;
                padding:20px;border-radius:4px;font-family:monospace;">
        <div style="color:#00d4ff;font-size:11px;letter-spacing:0.1em;text-transform:uppercase;
                    margin-bottom:12px;">Gemma 4 · RAG-Grounded Structural Interpretation</div>
        <div style="color:#b0c4d0;font-size:13px;line-height:1.8;">
    ''' + data['llm_narration'].replace('\n', '<br>') + '''
        </div>
    </div>
    '''))
else:
    print('No LLM narration available.')

# ── 7. Run Live Analysis (optional) ──────────────────────
# Uncomment to run fresh analysis from this notebook

# import sys
# sys.path.insert(0, '../..')
# from agents.structure_viz.agent import StructureVisualisationAgent
#
# pipeline_data = {
#     'accession': 'NM_000546',
#     'mutations': [
#         {'position': 524, 'original': 'G', 'mutant': 'A', 'amino_acid_change': 'R175H'},
#         {'position': 742, 'original': 'C', 'mutant': 'T', 'amino_acid_change': 'R248W'},
#     ]
# }
#
# agent = StructureVisualisationAgent(rag_chain=None)
# result = agent.run(pipeline_data)
# print('Analysis complete:', result['prediction_time_seconds'], 's')